# Analisis Deep Learning - Deteccion de Fraude\n\nEste notebook revisa los artefactos generados por `src/dl/train_dl.py`: curvas de entrenamiento, metricas, comparacion con ML y ejemplos de predicciones correctas e incorrectas.

In [ ]:
from pathlib import Path\nimport json\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nROOT = Path('..').resolve()\nREPORTS = ROOT / 'data' / 'reports'\nREPORTS

## Curvas de entrenamiento y validacion

In [ ]:
history_path = REPORTS / 'dl_training_history.csv'\n\nif history_path.exists():\n    history = pd.read_csv(history_path, index_col='epoch')\n    display(history.tail())\n\n    axes = history[['loss', 'val_loss']].plot(figsize=(10, 4), grid=True, title='Loss')\n    axes.set_xlabel('Epoca')\n    axes.set_ylabel('Binary crossentropy')\n    plt.show()\n\n    metric_cols = [c for c in ['auc', 'val_auc', 'recall', 'val_recall'] if c in history.columns]\n    if metric_cols:\n        ax = history[metric_cols].plot(figsize=(10, 4), grid=True, title='Metricas')\n        ax.set_xlabel('Epoca')\n        plt.show()\nelse:\n    print('Aun no existe dl_training_history.csv. Ejecuta: python src/dl/train_dl.py')

## Metricas finales del modelo DL

In [ ]:
metrics_path = REPORTS / 'dl_metrics.json'\n\nif metrics_path.exists():\n    with metrics_path.open() as f:\n        metrics = json.load(f)\n    display(pd.Series(metrics, name='deep_learning_mlp').to_frame())\nelse:\n    print('Aun no existe dl_metrics.json.')

## Comparacion con modelos de Machine Learning

In [ ]:
comparison_path = REPORTS / 'model_comparison.csv'\n\nif comparison_path.exists():\n    comparison = pd.read_csv(comparison_path, index_col=0)\n    display(comparison.sort_values('f1', ascending=False))\n\n    metric_subset = [c for c in ['precision', 'recall', 'f1', 'roc_auc', 'avg_precision'] if c in comparison.columns]\n    comparison[metric_subset].plot(kind='bar', figsize=(11, 5), grid=True, title='Comparacion cuantitativa')\n    plt.xticks(rotation=30, ha='right')\n    plt.ylim(0, 1.05)\n    plt.show()\nelse:\n    print('Aun no existe model_comparison.csv. Ejecuta primero los scripts de evaluacion.')

## Ejemplos de predicciones correctas e incorrectas

In [ ]:
examples_path = REPORTS / 'dl_prediction_examples.csv'\n\nif examples_path.exists():\n    examples = pd.read_csv(examples_path)\n    display(examples.groupby('result_type').size().rename('count').to_frame())\n    display(examples.sort_values(['result_type', 'fraud_probability'], ascending=[True, False]).head(20))\nelse:\n    print('Aun no existe dl_prediction_examples.csv.')

## Analisis de errores\n\nEn este dominio los falsos negativos son el caso mas critico, porque representan fraudes que el sistema deja pasar. Los falsos positivos tambien importan, pero su costo principal es friccion para el usuario. Si el modelo DL obtiene buen ROC-AUC pero bajo recall, conviene revisar el umbral de decision en vez de asumir que la arquitectura fallo.